# E5 - YOLO26s + SCConv trên DataSet2

Ablation độc lập: chỉ thay đúng hai `C3k2` ở nhánh neck tạo feature P3/8 và P4/16 bằng `C3k2_SCConv`. SCConv giữ chuỗi `SRU -> CRU`, dùng runtime graph của Ultralytics 8.4.155 và không đoán layer index.

Giữ nguyên dataset, class order, train/val split, augmentation, loss và protocol baseline. Không thêm P2, iAFF, MultiSEAM, DySample, SEAM, CBAM, ECA, AFPN, ADown, custom loss hoặc class weighting. Chỉ validation `split='val'`, output riêng và mọi metric/complexity đều đo thực tế.

In [ ]:
%pip install -q ultralytics==8.4.155
%pip install -q gdown
!apt-get update -qq
!apt-get install -y unrar

In [ ]:
from google.colab import drive
import os
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import ultralytics
import yaml
from PIL import Image
from ultralytics import YOLO

drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/DetectedHelmets/DataSet2', exist_ok=True)
os.chdir('/content/DetectedHelmets')
data_root = Path('/content/DetectedHelmets/DataSet2')
drive_dataset_path = Path('/content/drive/MyDrive/DataSet2')
if drive_dataset_path.exists():
    if data_root.is_symlink():
        data_root.unlink()
    elif data_root.exists() and data_root.resolve() != drive_dataset_path.resolve():
        shutil.rmtree(data_root)
    if not data_root.exists():
        data_root.symlink_to(drive_dataset_path, target_is_directory=True)
else:
    import gdown
    archive_path = Path('/content/DetectedHelmets/dataset.rar')
    file_id = '1LUUQLl5F3aczuny4As_PcTtqLIWTmZSJ'
    gdown.download(f'https://drive.google.com/uc?id={file_id}', str(archive_path), quiet=False, fuzzy=True)
    os.system(f'unrar x -o+ {archive_path} {data_root}/')

DATA_YAML_PATH = Path('/content/DetectedHelmets/DataSet2/DataSet2/data.yaml')
MODEL_NAME = 'yolo26s.pt'
ULTRALYTICS_VERSION = '8.4.155'
OUTPUT_DIR = Path('/content/drive/MyDrive/DetectedHelmets/runs/train')
EXPERIMENT_NAME = 'helmet_yolo26s_scconv_dataset2'
BASELINE_EXPERIMENT_NAME = 'helmet_yolo26s_dataset2'
EXPERIMENT_DIR = OUTPUT_DIR / EXPERIMENT_NAME
BEST_MODEL_PATH = EXPERIMENT_DIR / 'weights' / 'best.pt'
BASELINE_BEST_MODEL_PATH = OUTPUT_DIR / BASELINE_EXPERIMENT_NAME / 'weights' / 'best.pt'
VALIDATION_SPLIT = 'val'
TRAIN_CONFIG = {
    'epochs': 50, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4,
    'optimizer': 'auto', 'seed': 0, 'deterministic': True, 'patience': 15,
    'pretrained': True,
}

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Ultralytics:', ultralytics.__version__)
assert torch.cuda.is_available(), 'Protocol yêu cầu device=0 và GPU phải được bật.'
assert ultralytics.__version__ == ULTRALYTICS_VERSION
assert DATA_YAML_PATH.exists(), DATA_YAML_PATH
with DATA_YAML_PATH.open('r', encoding='utf-8') as file:
    data_config = yaml.safe_load(file)
raw_names = data_config.get('names', [])
if isinstance(raw_names, dict):
    raw_names = [raw_names[key] for key in sorted(raw_names, key=lambda item: int(item))]
CLASS_NAMES = list(raw_names)
EXPECTED_CLASS_NAMES = ['motorcycle', 'rider_with_helmet', 'rider_without_helmet']
assert data_config.get('nc') == 3
assert CLASS_NAMES == EXPECTED_CLASS_NAMES, CLASS_NAMES
print('Dataset:', DATA_YAML_PATH)
print('Classes:', CLASS_NAMES)
print('Protocol:', TRAIN_CONFIG)

## 1. Runtime graph, SCConv và unit checks

Cell này in architecture, chọn đúng input Detect P3/P4 từ runtime graph, implement `GroupBatchnorm2d`, `SRU`, `CRU`, `ScConv`, rồi unit-check shape và gradient. Nếu không có đúng một C3k2 neck target ở mỗi stride 8/16, notebook dừng và in architecture.

In [ ]:
base_model = YOLO(MODEL_NAME)
print('--- YOLO26s runtime architecture ---')
print(base_model.model)
base_model.info(verbose=True, imgsz=640)

def class_count(module, class_name):
    return sum(1 for item in module.modules() if item.__class__.__name__ == class_name)

def probe_c3k2(yolo_model, imgsz=640):
    records, handles = [], []
    def make_hook(index, module):
        def hook(_module, inputs, output):
            if not torch.is_tensor(output):
                return
            input_value = inputs[0] if inputs else None
            records.append({'index': index, 'from': getattr(module, 'f', None), 'input_shape': tuple(input_value.shape) if torch.is_tensor(input_value) else None, 'output_shape': tuple(output.shape), 'stride': imgsz // output.shape[-1], 'bottleneck_count': class_count(module, 'Bottleneck')})
        return hook
    for index, module in enumerate(yolo_model.model.model):
        if module.__class__.__name__ == 'C3k2':
            handles.append(module.register_forward_hook(make_hook(index, module)))
    device = next(yolo_model.model.parameters()).device
    was_training = yolo_model.model.training
    yolo_model.model.eval()
    with torch.no_grad():
        yolo_model.model(torch.zeros(1, 3, imgsz, imgsz, device=device))
    if was_training: yolo_model.model.train()
    for handle in handles: handle.remove()
    return sorted(records, key=lambda item: item['index'])
c3k2_probe = probe_c3k2(base_model)
for record in c3k2_probe: print(record)
detect_entries = [(i, m) for i, m in enumerate(base_model.model.model) if m.__class__.__name__ in {'Detect', 'WorldDetect', 'YOLOEDetect'}]
if not detect_entries:
    print(base_model.model)
    raise RuntimeError('Detect head not found.')
detect_index, detect_head = min(detect_entries, key=lambda item: item[0])
detect_from = getattr(detect_head, 'f', None)
if not isinstance(detect_from, (list, tuple)):
    print(base_model.model)
    raise RuntimeError('Detect input indices unavailable.')
detect_source_indices = {int(index) for index in detect_from}
upsample_indices = [i for i, m in enumerate(base_model.model.model) if m.__class__.__name__ == 'Upsample' and i < detect_index]
if not upsample_indices:
    print(base_model.model)
    raise RuntimeError('Neck start could not be found.')
neck_start_index = min(upsample_indices)
neck_candidates = [r for r in c3k2_probe if neck_start_index < r['index'] < detect_index and r['index'] in detect_source_indices and r['stride'] in (8, 16)]
selected_by_stride = {}
for stride in (8, 16):
    candidates = [r for r in neck_candidates if r['stride'] == stride]
    if len(candidates) != 1:
        print('Candidates:', candidates)
        print(base_model.model)
        raise RuntimeError('Expected exactly one C3k2 Detect input for stride {}.'.format(stride))
    selected_by_stride[stride] = candidates[0]
C3K2_SCCONV_EXPECTED_COUNT = 2
c3k2_specs = [{'level': 'P{}/{}'.format(stride // 8 + 2, stride), **selected_by_stride[stride]} for stride in (8, 16)]
print('Selected runtime C3k2 targets:', c3k2_specs)

def nearest_valid_group_num(c_num, requested):
    valid = [group for group in range(1, c_num + 1) if c_num % group == 0]
    return min(valid, key=lambda group: (abs(group - requested), -group))
class GroupBatchnorm2d(nn.Module):
    def __init__(self, c_num, group_num=4, eps=1e-5):
        super().__init__(); self.group_num = nearest_valid_group_num(c_num, group_num)
        if self.group_num != group_num: print('GroupBatchnorm2d adjusted group_num {} -> {} for C={}'.format(group_num, self.group_num, c_num))
        self.weight = nn.Parameter(torch.ones(c_num, 1, 1)); self.bias = nn.Parameter(torch.zeros(c_num, 1, 1)); self.eps = eps
    def forward(self, x):
        batch, channels, height, width = x.shape; grouped = x.reshape(batch, self.group_num, -1)
        mean = grouped.mean(2, keepdim=True); variance = grouped.var(2, keepdim=True, unbiased=False)
        normalized = ((grouped - mean) / torch.sqrt(variance + self.eps)).reshape(batch, channels, height, width)
        return normalized * self.weight + self.bias
class SRU(nn.Module):
    def __init__(self, oup_channels, group_num=4, gate_threshold=0.5):
        super().__init__()
        if oup_channels % 2: raise ValueError('SRU cross reconstruction requires even channels.')
        self.gn = GroupBatchnorm2d(oup_channels, group_num); self.gate_threshold = gate_threshold; self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        gn_x = self.gn(x); gamma = self.gn.weight; gamma_weight = (gamma / (gamma.sum() + torch.finfo(gamma.dtype).eps)).view(1, -1, 1, 1)
        weights = self.sigmoid(gn_x * gamma_weight)
        informative = torch.where(weights > self.gate_threshold, torch.ones_like(weights), weights)
        non_informative = torch.where(weights > self.gate_threshold, torch.zeros_like(weights), weights)
        return self.reconstruct(informative * x, non_informative * x)
    def reconstruct(self, x1, x2):
        half = x1.shape[1] // 2; x11, x12 = torch.split(x1, [half, half], 1); x21, x22 = torch.split(x2, [half, half], 1)
        return torch.cat([x11 + x22, x12 + x21], 1)
class CRU(nn.Module):
    def __init__(self, op_channel, alpha=0.5, squeeze_ratio=2, group_size=2, group_kernel_size=3):
        super().__init__(); self.up_channel = int(alpha * op_channel); self.low_channel = op_channel - self.up_channel
        self.up_squeeze = self.up_channel // squeeze_ratio; self.low_squeeze = self.low_channel // squeeze_ratio
        if min(self.up_squeeze, self.low_squeeze) < 1 or self.up_squeeze % group_size or op_channel % group_size: raise ValueError('CRU channel configuration is incompatible.')
        self.squeeze1 = nn.Conv2d(self.up_channel, self.up_squeeze, 1, bias=False); self.squeeze2 = nn.Conv2d(self.low_channel, self.low_squeeze, 1, bias=False)
        self.GWC = nn.Conv2d(self.up_squeeze, op_channel, group_kernel_size, 1, group_kernel_size // 2, groups=group_size)
        self.PWC1 = nn.Conv2d(self.up_squeeze, op_channel, 1, bias=False); self.PWC2 = nn.Conv2d(self.low_squeeze, op_channel - self.low_squeeze, 1, bias=False); self.advavg = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        upper, lower = torch.split(x, [self.up_channel, self.low_channel], 1); upper, lower = self.squeeze1(upper), self.squeeze2(lower)
        upper_branch = self.GWC(upper) + self.PWC1(upper); lower_branch = torch.cat([self.PWC2(lower), lower], 1)
        fused = torch.cat([upper_branch, lower_branch], 1); fused = F.softmax(self.advavg(fused), dim=1) * fused
        first, second = torch.split(fused, fused.shape[1] // 2, 1); return first + second
class ScConv(nn.Module):
    def __init__(self, op_channel, group_num=4, gate_threshold=0.5, alpha=0.5, squeeze_ratio=2, group_size=2, group_kernel_size=3):
        super().__init__(); self.SRU = SRU(op_channel, group_num, gate_threshold); self.CRU = CRU(op_channel, alpha, squeeze_ratio, group_size, group_kernel_size)
    def forward(self, x):
        x = self.SRU(x)
        x = self.CRU(x)
        return x
unit_input = torch.randn(2, 32, 16, 16, requires_grad=True); unit_module = ScConv(32); unit_output = unit_module(unit_input)
assert unit_output.shape == unit_input.shape; unit_output.mean().backward(); assert any(p.grad is not None for p in unit_module.parameters())
print('SCConv unit shape/gradient checks: OK')

## 2. C3k2_SCConv integration, preflight, trainer và checkpoint

In [ ]:
from ultralytics.nn.modules.block import Bottleneck
SCONV_CONFIG = {'group_num': 4, 'gate_threshold': 0.5, 'alpha': 0.5, 'squeeze_ratio': 2, 'group_size': 2, 'group_kernel_size': 3}
def wrapped_channels(wrapper):
    raw = getattr(wrapper, 'conv', wrapper); return int(raw.in_channels), int(raw.out_channels)
def wrapped_stride(wrapper):
    stride = getattr(getattr(wrapper, 'conv', wrapper), 'stride', 1); return (stride, stride) if isinstance(stride, int) else tuple(stride)
class Bottleneck_SCConv(nn.Module):
    def __init__(self, original):
        super().__init__(); self.cv1 = original.cv1; self.add = bool(getattr(original, 'add', False))
        input_channels, output_channels = wrapped_channels(original.cv2)
        self.scconv_applied = wrapped_stride(original.cv2) == (1, 1) and input_channels == output_channels and input_channels % 2 == 0
        self.cv2 = ScConv(input_channels, **SCONV_CONFIG) if self.scconv_applied else original.cv2
    def forward(self, x):
        output = self.cv2(self.cv1(x)); return x + output if self.add else output
def replace_bottlenecks(module, stats):
    if module.__class__.__name__ == 'Bottleneck':
        replacement = Bottleneck_SCConv(module); stats['seen'] += 1; stats['scconv'] += int(replacement.scconv_applied); stats['kept'] += int(not replacement.scconv_applied); return replacement
    for name, child in list(module.named_children()):
        replacement = replace_bottlenecks(child, stats)
        if replacement is not child: setattr(module, name, replacement)
    return module
class C3k2_SCConv(nn.Module):
    def __init__(self, original):
        super().__init__(); self.c, self.cv1, self.cv2 = original.c, original.cv1, original.cv2
        self.stats = {'seen': 0, 'scconv': 0, 'kept': 0}; self.m = replace_bottlenecks(original.m, self.stats)
    def forward(self, x):
        values = list(self.cv1(x).chunk(2, 1)); values.extend(block(values[-1]) for block in self.m); return self.cv2(torch.cat(values, 1))
    def forward_split(self, x):
        values = list(self.cv1(x).split((self.c, self.c), 1)); values.extend(block(values[-1]) for block in self.m); return self.cv2(torch.cat(values, 1))
def copy_graph_metadata(new, old, index, from_value):
    new.i = getattr(old, 'i', index); new.f = getattr(old, 'f', from_value); new.type = getattr(old, 'type', new.__class__.__name__); new.np = sum(p.numel() for p in new.parameters()); return new
def count_scconv(module): return sum(1 for item in module.modules() if item.__class__.__name__ == 'ScConv')
def count_c3k2_scconv(module): return sum(1 for item in module.modules() if item.__class__.__name__ == 'C3k2_SCConv')
def sum_tensors(value):
    if torch.is_tensor(value): return value.float().sum()
    if isinstance(value, (list, tuple)): return sum((sum_tensors(item) for item in value), torch.zeros((), device=value[0].device if value and torch.is_tensor(value[0]) else None))
    if isinstance(value, dict): return sum((sum_tensors(item) for item in value.values()), torch.zeros(()))
    return torch.zeros(())
custom_model = YOLO(MODEL_NAME); replacement_stats = []
for spec in c3k2_specs:
    old = custom_model.model.model[spec['index']]
    if old.__class__.__name__ != 'C3k2': print(custom_model.model); raise RuntimeError('Target is no longer C3k2.')
    new = C3k2_SCConv(old); replacement_stats.append((spec['level'], new.stats)); custom_model.model.model[spec['index']] = copy_graph_metadata(new, old, spec['index'], spec['from'])
source_state, target_state = base_model.model.state_dict(), custom_model.model.state_dict()
compatible_state = {key: value for key, value in source_state.items() if key in target_state and tuple(value.shape) == tuple(target_state[key].shape)}
load_report = custom_model.model.load_state_dict(compatible_state, strict=False)
source_parameters, target_parameters = dict(base_model.model.named_parameters()), dict(custom_model.model.named_parameters())
transferred_parameters = sum(source_parameters[key].numel() for key in compatible_state if key in source_parameters and key in target_parameters)
new_scconv_parameters = sum(p.numel() for item in custom_model.model.modules() if item.__class__.__name__ == 'ScConv' for p in item.parameters())
print('Replacement stats:', replacement_stats); print('Total pretrained parameters:', sum(p.numel() for p in base_model.model.parameters())); print('Transferred parameters:', transferred_parameters); print('New SCConv parameters:', new_scconv_parameters)
print('C3k2_SCConv:', count_c3k2_scconv(custom_model.model), 'ScConv:', count_scconv(custom_model.model)); assert count_c3k2_scconv(custom_model.model) == 2 and count_scconv(custom_model.model) > 0
shape_records, handles = {}, []
for spec in c3k2_specs:
    index = spec['index']
    def shape_hook(_module, inputs, output, index=index): shape_records[index] = (tuple(inputs[0].shape), tuple(output.shape))
    handles.append(custom_model.model.model[index].register_forward_hook(shape_hook))
device = next(custom_model.model.parameters()).device; custom_model.model.eval()
with torch.no_grad(): dummy_output = custom_model.model(torch.randn(2, 3, 640, 640, device=device))
for handle in handles: handle.remove()
assert any(torch.is_tensor(item) for item in (dummy_output if isinstance(dummy_output, (list, tuple)) else [dummy_output]))
for spec in c3k2_specs:
    observed_input, observed_output = shape_records[spec['index']]; assert observed_input[1:] == spec['input_shape'][1:]; assert observed_output[1:] == spec['output_shape'][1:]
print('Dummy forward and target input/output shapes: OK', shape_records)
custom_model.model.train(); custom_model.model.zero_grad(set_to_none=True)
graph_output = custom_model.model(torch.randn(2, 3, 640, 640, device=device, requires_grad=True)); graph_loss = sum_tensors(graph_output); assert graph_loss.requires_grad; graph_loss.backward()
scconv_grad_ok = any(p.grad is not None for item in custom_model.model.modules() if item.__class__.__name__ == 'ScConv' for p in item.parameters()); print('SCConv gradient OK:', scconv_grad_ok); assert scconv_grad_ok; custom_model.model.eval()
from ultralytics.models.yolo.detect import DetectionTrainer
custom_pytorch_model = custom_model.model
class CustomSCConvTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True): return custom_pytorch_model
    def setup_model(self):
        self.model = custom_pytorch_model; assert count_c3k2_scconv(self.model) == 2; print('C3k2_SCConv INSIDE setup_model:', count_c3k2_scconv(self.model)); return None
TRAIN_OVERRIDES = {'model': MODEL_NAME, 'data': str(DATA_YAML_PATH), **TRAIN_CONFIG, 'project': str(OUTPUT_DIR), 'name': EXPERIMENT_NAME, 'save': True, 'save_period': 10, 'plots': True, 'exist_ok': False}
trainer = CustomSCConvTrainer(overrides=TRAIN_OVERRIDES, _callbacks=custom_model.callbacks); trainer.model = custom_pytorch_model
c3k2_scconv_before_train, c3k2_scconv_inside_trainer = count_c3k2_scconv(custom_pytorch_model), count_c3k2_scconv(trainer.model); print('C3k2_SCConv BEFORE TRAIN:', c3k2_scconv_before_train); print('C3k2_SCConv INSIDE TRAINER:', c3k2_scconv_inside_trainer); assert c3k2_scconv_before_train == c3k2_scconv_inside_trainer == 2
train_results = trainer.train()
EXPERIMENT_VERSION_FILE = EXPERIMENT_DIR / 'ultralytics_version.txt'; EXPERIMENT_VERSION_FILE.write_text('ultralytics==' + ultralytics.__version__ + chr(10) + 'model=' + MODEL_NAME + chr(10) + 'c3k2_scconv_expected_count=2' + chr(10), encoding='utf-8')
def torch_load_checkpoint(path):
    try: return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError: return torch.load(path, map_location='cpu')
def extract_checkpoint_state(payload):
    if isinstance(payload, nn.Module): return payload.state_dict()
    if isinstance(payload, dict):
        for key in ('model', 'ema', 'state_dict'):
            value = payload.get(key)
            if isinstance(value, nn.Module): return value.state_dict()
            if isinstance(value, dict):
                tensors = {name: item for name, item in value.items() if torch.is_tensor(item)}
                if tensors: return tensors
        tensors = {name: item for name, item in payload.items() if torch.is_tensor(item)}
        if tensors: return tensors
    raise TypeError('Checkpoint state dict not found.')
def load_custom_scconv_checkpoint(path):
    try:
        direct = YOLO(str(path))
        if count_c3k2_scconv(direct.model) == 2: return direct, 'direct'
    except Exception as direct_error: print('Direct load unavailable:', direct_error)
    rebuilt = YOLO(MODEL_NAME)
    for spec in c3k2_specs:
        old = rebuilt.model.model[spec['index']]; rebuilt.model.model[spec['index']] = copy_graph_metadata(C3k2_SCConv(old), old, spec['index'], spec['from'])
    state = extract_checkpoint_state(torch_load_checkpoint(str(path)))
    if all(name.startswith('module.') for name in state): state = {name[7:]: value for name, value in state.items()}
    target = rebuilt.model.state_dict(); compatible = {name: value for name, value in state.items() if name in target and tuple(value.shape) == tuple(target[name].shape)}
    if not compatible: raise RuntimeError('No compatible checkpoint tensors.')
    report = rebuilt.model.load_state_dict(compatible, strict=False); print('Rebuilt compatible tensors:', len(compatible), 'missing:', len(report.missing_keys)); assert count_c3k2_scconv(rebuilt.model) == 2; return rebuilt, 'rebuilt'
if not BEST_MODEL_PATH.exists(): raise FileNotFoundError(BEST_MODEL_PATH)
best_model, checkpoint_load_mode = load_custom_scconv_checkpoint(BEST_MODEL_PATH); c3k2_scconv_count_best, scconv_count_best = count_c3k2_scconv(best_model.model), count_scconv(best_model.model)
print('C3k2_SCConv IN BEST MODEL:', c3k2_scconv_count_best); print('ScConv IN BEST MODEL:', scconv_count_best, 'load mode:', checkpoint_load_mode); assert c3k2_scconv_count_best == 2 and scconv_count_best > 0

## 3. Validation metrics, plots, error analysis, complexity và so sánh baseline

In [ ]:
def safe_f1(precision, recall): return 2 * precision * recall / (precision + recall) if precision + recall else 0.0
def extract_metrics(result, label):
    summary = {'Precision': float(result.box.mp), 'Recall': float(result.box.mr), 'mAP50': float(result.box.map50), 'mAP50-95': float(result.box.map)}; summary['F1'] = safe_f1(summary['Precision'], summary['Recall']); summary['by_class'] = {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        precision, recall, ap50, ap5095 = result.box.class_result(class_id); values = {'Precision': float(precision), 'Recall': float(recall), 'AP50': float(ap50), 'AP50-95': float(ap5095)}; values['F1'] = safe_f1(values['Precision'], values['Recall']); summary['by_class'][class_name] = values
    print(label, summary); return summary
validation_metrics = best_model.val(data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=True)
validation_summary = extract_metrics(validation_metrics, 'E5 validation'); no_helmet_metrics = validation_summary['by_class']['rider_without_helmet']; test_metrics = None; test_summary = None
plot_specs = [('results.png', 'Training results'), ('confusion_matrix.png', 'Confusion matrix'), ('confusion_matrix_normalized.png', 'Normalized confusion matrix'), ('BoxPR_curve.png', 'Box PR curve'), ('BoxF1_curve.png', 'Box F1 curve'), ('BoxP_curve.png', 'Box Precision curve'), ('BoxR_curve.png', 'Box Recall curve')]
for filename, title in plot_specs:
    paths = list(EXPERIMENT_DIR.rglob(filename)) if EXPERIMENT_DIR.exists() else []
    if paths:
        path = max(paths, key=lambda item: item.stat().st_mtime); plt.figure(figsize=(16, 10) if filename == 'results.png' else (10, 8)); plt.imshow(Image.open(path)); plt.axis('off'); plt.title(title); plt.show()
def confusion_array(result):
    matrix = getattr(getattr(result, 'confusion_matrix', None), 'matrix', None); matrix = None if matrix is None else np.asarray(matrix); return matrix if matrix is not None and matrix.shape == (len(CLASS_NAMES) + 1, len(CLASS_NAMES) + 1) else None
def error_counts(result):
    matrix = confusion_array(result)
    if matrix is None: return None
    labels = CLASS_NAMES + ['background']; index = {label: i for i, label in enumerate(labels)}
    return {'rider_without_helmet -> rider_with_helmet': float(matrix[index['rider_with_helmet'], index['rider_without_helmet']]), 'rider_without_helmet -> background': float(matrix[index['background'], index['rider_without_helmet']]), 'background -> rider_without_helmet': float(matrix[index['rider_without_helmet'], index['background']])}
scconv_errors = error_counts(validation_metrics)
if scconv_errors is not None: display(pd.DataFrame({'Error': list(scconv_errors), 'Count': list(scconv_errors.values())}))
valid_img_dir = Path('/content/DetectedHelmets/DataSet2/DataSet2/valid/images'); valid_images = sorted(str(path) for pattern in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG') for path in valid_img_dir.glob(pattern)); assert valid_images
def extract_gflops(info):
    if isinstance(info, dict):
        for key in ('GFLOPs', 'gflops', 'flops'):
            if key in info:
                try: return float(info[key])
                except (TypeError, ValueError): pass
    for value in reversed(list(info) if isinstance(info, (list, tuple)) else []):
        try: return float(value)
        except (TypeError, ValueError): pass
    return None
def benchmark(model, images):
    for image in images[:2]: model.predict(source=image, imgsz=640, device=0, save=False, verbose=False)
    elapsed = []
    for image in images:
        torch.cuda.synchronize(); start = time.perf_counter(); model.predict(source=image, imgsz=640, device=0, save=False, verbose=False); torch.cuda.synchronize(); elapsed.append(time.perf_counter() - start)
    milliseconds = float(np.mean(elapsed) * 1000); return milliseconds, 1000 / milliseconds
assert BASELINE_BEST_MODEL_PATH.exists(); baseline_model = YOLO(str(BASELINE_BEST_MODEL_PATH)); speed_images = valid_images[:min(50, len(valid_images))]; baseline_ms, baseline_fps = benchmark(baseline_model, speed_images); scconv_ms, scconv_fps = benchmark(best_model, speed_images)
complexity = pd.DataFrame([{'Model': 'YOLO26s', 'Parameters': sum(p.numel() for p in baseline_model.model.parameters()), 'GFLOPs': extract_gflops(baseline_model.info(verbose=True, imgsz=640)), 'Model size (MB)': BASELINE_BEST_MODEL_PATH.stat().st_size / 1024 ** 2, 'Inference (ms/image)': baseline_ms, 'FPS': baseline_fps}, {'Model': 'YOLO26s+SCConv', 'Parameters': sum(p.numel() for p in best_model.model.parameters()), 'GFLOPs': extract_gflops(best_model.info(verbose=True, imgsz=640)), 'Model size (MB)': BEST_MODEL_PATH.stat().st_size / 1024 ** 2, 'Inference (ms/image)': scconv_ms, 'FPS': scconv_fps}]); display(complexity)
baseline_metrics = baseline_model.val(data=str(DATA_YAML_PATH), split=VALIDATION_SPLIT, imgsz=640, device=0, plots=False)
baseline_summary = extract_metrics(baseline_metrics, 'YOLO26s baseline')
def comparison_values(summary):
    nh = summary['by_class']['rider_without_helmet']; return {'Precision': summary['Precision'], 'Recall': summary['Recall'], 'F1': summary['F1'], 'mAP50': summary['mAP50'], 'mAP50-95': summary['mAP50-95'], 'NoHelmet Precision': nh['Precision'], 'NoHelmet Recall': nh['Recall'], 'NoHelmet F1': nh['F1'], 'NoHelmet AP50': nh['AP50'], 'NoHelmet AP50-95': nh['AP50-95']}
baseline_values, scconv_values = comparison_values(baseline_summary), comparison_values(validation_summary); comparison = pd.DataFrame({'Metric': list(baseline_values), 'YOLO26s': list(baseline_values.values()), 'YOLO26s+SCConv': [scconv_values[key] for key in baseline_values]}); comparison['Delta'] = comparison['YOLO26s+SCConv'] - comparison['YOLO26s']; display(comparison)
baseline_errors = error_counts(baseline_metrics)
if baseline_errors is not None and scconv_errors is not None:
    error_comparison = pd.DataFrame({'Error': list(baseline_errors), 'YOLO26s': list(baseline_errors.values()), 'YOLO26s+SCConv': [scconv_errors[key] for key in baseline_errors]}); error_comparison['Delta'] = error_comparison['YOLO26s+SCConv'] - error_comparison['YOLO26s']; display(error_comparison)
priority = ['mAP50-95', 'NoHelmet AP50', 'NoHelmet Recall']; priority_improvements = {key: scconv_values[key] > baseline_values[key] for key in priority}; joint_priority_drop = all(scconv_values[key] < baseline_values[key] for key in priority)
error_background_reduced = baseline_errors is not None and scconv_errors is not None and scconv_errors['rider_without_helmet -> background'] < baseline_errors['rider_without_helmet -> background']
params_reduced = complexity.iloc[1]['Parameters'] < complexity.iloc[0]['Parameters']; gflops_reduced = complexity.iloc[0]['GFLOPs'] is not None and complexity.iloc[1]['GFLOPs'] is not None and complexity.iloc[1]['GFLOPs'] < complexity.iloc[0]['GFLOPs']
e5_decision = 'KEEP SCConv' if ((any(priority_improvements.values()) or error_background_reduced) and not joint_priority_drop) or (not joint_priority_drop and params_reduced and gflops_reduced) else 'REJECT SCConv'
print('Priority improvements:', priority_improvements); print('NoHelmet -> background reduced:', error_background_reduced); print('E5 decision:', e5_decision)

## 4. Final audit

In [ ]:
assert ultralytics.__version__ == '8.4.155'
assert CLASS_NAMES == EXPECTED_CLASS_NAMES
assert TRAIN_CONFIG == {'epochs': 50, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4, 'optimizer': 'auto', 'seed': 0, 'deterministic': True, 'patience': 15, 'pretrained': True}
assert VALIDATION_SPLIT == 'val' and len(c3k2_specs) == 2 and {spec['stride'] for spec in c3k2_specs} == {8, 16}
assert trainer.model is custom_pytorch_model
assert c3k2_scconv_before_train == c3k2_scconv_inside_trainer == c3k2_scconv_count_best == 2
assert scconv_grad_ok and scconv_count_best > 0 and test_metrics is None and test_summary is None
assert EXPERIMENT_NAME == 'helmet_yolo26s_scconv_dataset2' and BASELINE_EXPERIMENT_NAME not in str(BEST_MODEL_PATH)
forbidden = {'P2', 'iAFF', 'MultiSEAM', 'DySample', 'SEAM', 'CBAM', 'ECA', 'AFPN', 'ADown'}; custom_names = {item.__class__.__name__ for item in best_model.model.modules()}; assert not custom_names.intersection(forbidden)
assert EXPERIMENT_VERSION_FILE.exists()
print('Final audit passed.'); print('Ultralytics:', ultralytics.__version__); print('C3k2_SCConv before/train/best:', c3k2_scconv_before_train, c3k2_scconv_inside_trainer, c3k2_scconv_count_best); print('ScConv in best model:', scconv_count_best); print('Validation only; test disabled; output:', BEST_MODEL_PATH); print('Decision:', e5_decision)